# P-021: Panel Sensitivity to Model Hyperparameters

**Decision question:** Is the panel robust to reasonable hyperparameter changes, or is it an artifact of the exact locked ExtraTrees configuration?

**Approach:** Sweep a 5x5 grid of `n_estimators` x `max_features` (25 configs), each evaluated over 10 random 80/20 splits. Track panel devices' top-20 appearance rates and tau-b across all configs.

**Panel:** Device 850, 1320, 119

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau

# ── Load data ──
df = pd.read_csv("perovskite_stability_clean.csv").fillna(0)

FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature",
    "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]
TARGET = "Stability_PCE_T80"
PANEL = [850, 1320, 119]

X = df[FEATURES].values
y = np.log1p(df[TARGET].values)

print(f"Dataset: {X.shape[0]} devices, {X.shape[1]} features")
print(f"Panel indices: {PANEL}")

# ── Hyperparameter grid ──
n_estimators_grid = [100, 300, 500, 700, 1000]
max_features_grid = ['sqrt', 'log2', 0.5, 0.75, 1.0]

configs = []
for ne in n_estimators_grid:
    for mf in max_features_grid:
        configs.append({"n_estimators": ne, "max_features": mf})

print(f"\nHyperparameter grid: {len(n_estimators_grid)} x {len(max_features_grid)} = {len(configs)} configs")
print(f"Splits per config: 10  |  Total model fits: {len(configs) * 10}")

Dataset: 1543 devices, 16 features
Panel indices: [850, 1320, 119]

Hyperparameter grid: 5 x 5 = 25 configs
Splits per config: 10  |  Total model fits: 250


In [2]:
import time

N_SPLITS = 10
TOP_K = 20

# Results storage
results = []  # one row per config

t0 = time.time()

for ci, cfg in enumerate(configs):
    tau_list = []
    # panel_hits[dev] = number of splits where dev lands in top-20
    panel_hits = {dev: 0 for dev in PANEL}
    panel_appearances = {dev: 0 for dev in PANEL}  # times dev is in test set

    for seed in range(N_SPLITS):
        idx = np.arange(len(y))
        train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=seed)

        et = ExtraTreesRegressor(
            n_estimators=cfg["n_estimators"],
            max_features=cfg["max_features"],
            min_samples_split=3,
            min_samples_leaf=1,
            bootstrap=False,
            random_state=42,
        )
        et.fit(X[train_idx], y[train_idx])
        preds = et.predict(X[test_idx])

        # tau-b on test set
        tau, _ = kendalltau(y[test_idx], preds)
        tau_list.append(tau)

        # Rank test devices by predicted stability (descending)
        order = np.argsort(-preds)
        top_indices = test_idx[order[:TOP_K]]

        for dev in PANEL:
            if dev in test_idx:
                panel_appearances[dev] += 1
                if dev in top_indices:
                    panel_hits[dev] += 1

    # Top-20 rate per panel device
    panel_rates = {}
    for dev in PANEL:
        if panel_appearances[dev] > 0:
            panel_rates[dev] = panel_hits[dev] / panel_appearances[dev]
        else:
            panel_rates[dev] = np.nan

    row = {
        "n_estimators": cfg["n_estimators"],
        "max_features": cfg["max_features"],
        "mean_tau": np.mean(tau_list),
        "std_tau": np.std(tau_list),
    }
    for dev in PANEL:
        row[f"dev{dev}_top20_rate"] = panel_rates[dev]
        row[f"dev{dev}_appearances"] = panel_appearances[dev]
    results.append(row)

    if (ci + 1) % 5 == 0:
        elapsed = time.time() - t0
        print(f"  Config {ci+1}/{len(configs)} done  ({elapsed:.1f}s elapsed)")

elapsed = time.time() - t0
print(f"\nAll {len(configs)} configs complete in {elapsed:.1f}s")

results_df = pd.DataFrame(results)
print(f"Results table: {results_df.shape}")

  Config 5/25 done  (5.0s elapsed)


  Config 10/25 done  (20.0s elapsed)


  Config 15/25 done  (45.2s elapsed)


  Config 20/25 done  (80.9s elapsed)


  Config 25/25 done  (132.2s elapsed)

All 25 configs complete in 132.2s
Results table: (25, 10)


In [3]:
# ── Results table sorted by tau-b ──
display_cols = ["n_estimators", "max_features", "mean_tau", "std_tau",
                "dev850_top20_rate", "dev1320_top20_rate", "dev119_top20_rate"]

sorted_df = results_df[display_cols].sort_values("mean_tau", ascending=False).reset_index(drop=True)
sorted_df.index.name = "rank"

print("=" * 90)
print("Full results: 25 configs sorted by mean tau-b (descending)")
print("=" * 90)
with pd.option_context("display.max_rows", 30, "display.float_format", "{:.4f}".format,
                        "display.max_columns", 10, "display.width", 120):
    print(sorted_df.to_string())

print(f"\nBest tau-b:  {sorted_df['mean_tau'].iloc[0]:.4f}  "
      f"(n_est={sorted_df['n_estimators'].iloc[0]}, max_feat={sorted_df['max_features'].iloc[0]})")
print(f"Worst tau-b: {sorted_df['mean_tau'].iloc[-1]:.4f}  "
      f"(n_est={sorted_df['n_estimators'].iloc[-1]}, max_feat={sorted_df['max_features'].iloc[-1]})")

Full results: 25 configs sorted by mean tau-b (descending)
      n_estimators max_features  mean_tau  std_tau  dev850_top20_rate  dev1320_top20_rate  dev119_top20_rate
rank                                                                                                        
0              300         sqrt    0.3004   0.0434             1.0000              1.0000             1.0000
1              300         log2    0.3004   0.0434             1.0000              1.0000             1.0000
2              500         sqrt    0.3002   0.0445             1.0000              1.0000             1.0000
3              500         log2    0.3002   0.0445             1.0000              1.0000             1.0000
4              700         sqrt    0.3000   0.0448             1.0000              1.0000             1.0000
5              700         log2    0.3000   0.0448             1.0000              1.0000             1.0000
6             1000         log2    0.3000   0.0446             1.0000

In [4]:
# ── Panel survival analysis ──
# For each panel device, what fraction of the 25 configs give >= 80% top-20 rate?

SURVIVAL_THRESHOLD = 0.80

print("=" * 70)
print("Panel Survival Analysis")
print(f"Threshold: top-20 rate >= {SURVIVAL_THRESHOLD:.0%} to count as 'surviving'")
print("=" * 70)

for dev in PANEL:
    col = f"dev{dev}_top20_rate"
    rates = results_df[col].dropna()
    n_survive = (rates >= SURVIVAL_THRESHOLD).sum()
    frac = n_survive / len(rates) if len(rates) > 0 else 0
    print(f"\n  Device {dev}:")
    print(f"    Top-20 rates across configs: min={rates.min():.2%}, "
          f"median={rates.median():.2%}, max={rates.max():.2%}")
    print(f"    Configs with >= {SURVIVAL_THRESHOLD:.0%} top-20 rate: "
          f"{n_survive}/{len(rates)} ({frac:.0%})")

# Summary
print("\n" + "-" * 70)
survival_fracs = {}
for dev in PANEL:
    col = f"dev{dev}_top20_rate"
    rates = results_df[col].dropna()
    survival_fracs[dev] = (rates >= SURVIVAL_THRESHOLD).mean()
    
print("Panel survival fractions (configs with >= 80% top-20 rate):")
for dev, frac in survival_fracs.items():
    tag = "ROBUST" if frac >= 0.80 else ("MODERATE" if frac >= 0.60 else "FRAGILE")
    print(f"  Device {dev}: {frac:.0%}  [{tag}]")

Panel Survival Analysis
Threshold: top-20 rate >= 80% to count as 'surviving'

  Device 850:
    Top-20 rates across configs: min=100.00%, median=100.00%, max=100.00%
    Configs with >= 80% top-20 rate: 25/25 (100%)

  Device 1320:
    Top-20 rates across configs: min=100.00%, median=100.00%, max=100.00%
    Configs with >= 80% top-20 rate: 25/25 (100%)

  Device 119:
    Top-20 rates across configs: min=100.00%, median=100.00%, max=100.00%
    Configs with >= 80% top-20 rate: 25/25 (100%)

----------------------------------------------------------------------
Panel survival fractions (configs with >= 80% top-20 rate):
  Device 850: 100%  [ROBUST]
  Device 1320: 100%  [ROBUST]
  Device 119: 100%  [ROBUST]


In [5]:
# ── Worst reasonable config analysis ──
# "Worst reasonable" = lowest tau-b that's still within 10% of the locked config's tau-b

# Find locked config tau-b
locked_mask = (results_df["n_estimators"] == 700) & (results_df["max_features"] == "sqrt")
locked_tau = results_df.loc[locked_mask, "mean_tau"].values[0]
threshold_tau = locked_tau * 0.90  # within 10%

print(f"Locked config (n_est=700, max_feat=sqrt) tau-b: {locked_tau:.4f}")
print(f"10% threshold: tau-b >= {threshold_tau:.4f}")
print()

# Filter to reasonable configs
reasonable = results_df[results_df["mean_tau"] >= threshold_tau].copy()
worst_idx = reasonable["mean_tau"].idxmin()
worst = reasonable.loc[worst_idx]

print(f"Worst reasonable config:")
print(f"  n_estimators = {int(worst['n_estimators'])}")
print(f"  max_features = {worst['max_features']}")
print(f"  mean tau-b   = {worst['mean_tau']:.4f}")
print()

print("Panel device performance under worst reasonable config:")
for dev in PANEL:
    rate = worst[f"dev{dev}_top20_rate"]
    status = "SURVIVES" if rate >= 0.80 else ("MARGINAL" if rate >= 0.60 else "FAILS")
    print(f"  Device {dev}: top-20 rate = {rate:.0%}  [{status}]")

print(f"\n{len(reasonable)} of {len(results_df)} configs are within 10% of locked tau-b")

Locked config (n_est=700, max_feat=sqrt) tau-b: 0.3000
10% threshold: tau-b >= 0.2700

Worst reasonable config:
  n_estimators = 100
  max_features = 1.0
  mean tau-b   = 0.2918

Panel device performance under worst reasonable config:
  Device 850: top-20 rate = 100%  [SURVIVES]
  Device 1320: top-20 rate = 100%  [SURVIVES]
  Device 119: top-20 rate = 100%  [SURVIVES]

25 of 25 configs are within 10% of locked tau-b


In [6]:
# ── Save results ──
out_path = "Packet_P021_Hyperparameter_Sensitivity.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({results_df.shape[0]} rows x {results_df.shape[1]} cols)")

Saved: Packet_P021_Hyperparameter_Sensitivity.csv  (25 rows x 10 cols)


In [7]:
# ── P-021 Status Footer ──
print("=" * 70)
print("P-021  PANEL SENSITIVITY TO MODEL HYPERPARAMETERS")
print("=" * 70)

# Criteria:
# Confirmed: all 3 panel devices survive (>=80% top-20) in >=80% of configs
# Promising: >=2 devices survive in >=60% of configs
# Negative: panel is config-dependent

n_robust = sum(1 for f in survival_fracs.values() if f >= 0.80)
n_moderate = sum(1 for f in survival_fracs.values() if f >= 0.60)

print(f"\nPanel devices with ROBUST survival (>=80% of configs): {n_robust}/3")
print(f"Panel devices with MODERATE+ survival (>=60% of configs): {n_moderate}/3")
print()

for dev in PANEL:
    print(f"  Device {dev}: survives in {survival_fracs[dev]:.0%} of configs")

print()

if n_robust == 3:
    status = "CONFIRMED"
    detail = ("All 3 panel devices maintain >=80% top-20 rate across >=80% of "
              "hyperparameter configs. The panel is NOT an artifact of the locked config.")
elif n_moderate >= 2:
    status = "PROMISING"
    detail = (f"{n_moderate}/3 panel devices survive in >=60% of configs. "
              "Panel is mostly robust but some devices are sensitive to hyperparameters.")
else:
    status = "NEGATIVE"
    detail = "Panel composition depends on specific hyperparameter choices."

print(f"P-021 status: ** {status} **")
print(f"  {detail}")
print()
print(f"Locked config tau-b: {locked_tau:.4f}")
print(f"Tau-b range across grid: [{results_df['mean_tau'].min():.4f}, {results_df['mean_tau'].max():.4f}]")

P-021  PANEL SENSITIVITY TO MODEL HYPERPARAMETERS

Panel devices with ROBUST survival (>=80% of configs): 3/3
Panel devices with MODERATE+ survival (>=60% of configs): 3/3

  Device 850: survives in 100% of configs
  Device 1320: survives in 100% of configs
  Device 119: survives in 100% of configs

P-021 status: ** CONFIRMED **
  All 3 panel devices maintain >=80% top-20 rate across >=80% of hyperparameter configs. The panel is NOT an artifact of the locked config.

Locked config tau-b: 0.3000
Tau-b range across grid: [0.2918, 0.3004]
